# Stride-8 FPV training (docs/DECISIONS.md #26) — RESEARCH / FEASIBILITY ONLY

Trains `config/stride8_fpv.yaml` (SSDLite + stride-8 level, geometric anchors, 640) on the
VisioDECT DJIFPV train blocks. VisioDECT's license is UNVERIFIED: these weights must never
enter a `--commercial-only` model. `training_report.json` records `commercially_cleared: false`.

**Before running**, build the bundle in the repo checkout and upload it to Drive as
`MyDrive/zafar/stride8_fpv_bundle.tar.gz`:

```
git ls-files -co --exclude-standard | grep -v '^data/\|^models/' > /tmp/files.txt
tar -czhf stride8_fpv_bundle.tar.gz -T /tmp/files.txt \
    data/visiodect/fpv-split/train data/visiodect/fpv-split/val \
    data/visiodect/fpv-split/split_manifest.json data/frozen_eval/visiodect-fpv-test
```

Checkpoints go to Drive every epoch. **If the runtime disconnects, re-run all cells**: training
resumes from the last completed epoch (`checkpoint.pt`), with the same seed and RNG state.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DRIVE = '/content/drive/MyDrive/zafar'
OUT = f'{DRIVE}/models/stride8_fpv'
!mkdir -p /content/work && tar -xzf {DRIVE}/stride8_fpv_bundle.tar.gz -C /content/work
%cd /content/work
!pip -q install pyyaml opencv-python-headless
!python -c "import torch, torchvision; print(torch.__version__, torchvision.__version__, torch.cuda.is_available())"

Anchor check on the real model: on the train blocks it must print `all | 1900 | 69.5% (0.562)`.

In [ ]:
!python -m eval.anchor_coverage --dataset-dir data/visiodect/fpv-split/train --train-config config/stride8_fpv.yaml

Train. Re-running this cell resumes. Early stopping: 5 epochs without a new best val AP@0.5.

In [ ]:
!python detector/train.py --config config/stride8_fpv.yaml --output-dir {OUT} 2>&1 | grep -v '^ ' | tail -40

In [ ]:
import json
r = json.load(open(f'{OUT}/training_report.json'))
print('status', r['status'], '| epochs', r['epochs_completed'], '| stopped early', r['stopped_early'],
      '| best val epoch', r['best_val_epoch'], '| device', r['device'])
print('provenance', r['provenance'])
print('leakage guard checked', r['leakage_guard_images_checked'], 'eval images in', r['leakage_guard_eval_dirs'])
for h in r['val_per_epoch']:
    print(h['epoch'], 'val AP@0.5', h['ap50'], 'raw scores', h['raw_score_min'], '-', h['raw_score_max'])

**Control run**: stock SSDLite 320, same split / seed / schedule (`config/stock320_fpv_control.yaml`). Re-running resumes.

In [ ]:
CTRL = f'{DRIVE}/models/stock320_fpv_control'
!python detector/train.py --config config/stock320_fpv_control.yaml --output-dir {CTRL} 2>&1 | grep -v '^ ' | tail -40

In [ ]:
r = json.load(open(f'{CTRL}/training_report.json'))
print('status', r['status'], '| epochs', r['epochs_completed'], '| best val epoch', r['best_val_epoch'])
for h in r['val_per_epoch']:
    print(h['epoch'], 'val AP@0.5', h['ap50'])

Download `weights_best.pt` and `training_report.json` from **both** `MyDrive/zafar/models/stride8_fpv/`
and `MyDrive/zafar/models/stock320_fpv_control/` into the repo's `models/stride8_fpv/` and
`models/stock320_fpv_control/`. Evaluation and FPS run there, on the same CPU as the dut_v1 baselines.